# Notebook 09 — Integrated Optimization to Physical Design Summary

This notebook integrates the earlier workflows into a single end-to-end design pipeline.

It focuses on:

- scanning a combined voltage-plus-geometry design space
- selecting a best design using a curvature-versus-height objective
- converting the selected design into physical units
- extracting trap height, pseudopotential scale, and secular-frequency estimates
- producing a compact hardware-facing design summary

This remains a simplified 2D model, but it shows the full logic chain from electrostatic design scan to physical trapped-ion design estimates.


## Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve


## Physical and numerical settings


In [ ]:
# Grid
nx, ny = 121, 81
x = np.linspace(-3.0, 3.0, nx)
y = np.linspace(0.0, 4.0, ny)
dx = x[1] - x[0]
dy = y[1] - y[0]

# Fixed side-electrode width
side_width = 1.3

# Physical scaling assumptions
e_charge = 1.602176634e-19          # C
amu = 1.66053906660e-27             # kg
ion_mass = 40.0 * amu               # 40Ca+
rf_frequency_hz = 20e6              # 20 MHz
Omega = 2.0 * np.pi * rf_frequency_hz
length_scale_m = 30e-6              # 30 um per notebook length unit

dx_phys = dx * length_scale_m
dy_phys = dy * length_scale_m

print(f'Grid: {nx} x {ny}')
print(f'dx = {dx:.3f}, dy = {dy:.3f}')
print(f'Length scale = {length_scale_m*1e6:.1f} um per notebook unit')
print(f'Ion mass = {ion_mass:.3e} kg')
print(f'RF drive frequency = {rf_frequency_hz/1e6:.1f} MHz')


## Helper functions


In [ ]:
def idx(i, j, nx):
    return i * nx + j

def make_electrode_masks(center_width, gap, side_width):
    center_half = center_width / 2.0
    left_center = -(center_half + gap + side_width / 2.0)
    right_center = +(center_half + gap + side_width / 2.0)

    left_mask = (x >= left_center - side_width / 2.0) & (x <= left_center + side_width / 2.0)
    center_mask = (x >= -center_half) & (x <= center_half)
    right_mask = (x >= right_center - side_width / 2.0) & (x <= right_center + side_width / 2.0)
    remaining_ground = ~(left_mask | center_mask | right_mask)
    return left_mask, center_mask, right_mask, remaining_ground

def build_boundary_problem(left_mask, center_mask, right_mask, remaining_ground, v_rf, v_dc):
    V = np.zeros((ny, nx), dtype=float)
    fixed = np.zeros((ny, nx), dtype=bool)

    # Ground outer boundaries
    V[:, 0] = 0.0
    V[:, -1] = 0.0
    V[-1, :] = 0.0
    fixed[:, 0] = True
    fixed[:, -1] = True
    fixed[-1, :] = True

    # Lower boundary electrodes
    bottom = 0
    V[bottom, left_mask] = v_dc
    V[bottom, center_mask] = v_rf
    V[bottom, right_mask] = v_dc
    V[bottom, remaining_ground] = 0.0
    fixed[bottom, :] = True
    return V, fixed

def solve_laplace(V, fixed):
    N = nx * ny
    A = lil_matrix((N, N), dtype=float)
    b = np.zeros(N, dtype=float)
    cx = 1.0 / dx**2
    cy = 1.0 / dy**2
    c0 = -2.0 * (cx + cy)

    for i in range(ny):
        for j in range(nx):
            k = idx(i, j, nx)
            if fixed[i, j]:
                A[k, k] = 1.0
                b[k] = V[i, j]
            else:
                A[k, idx(i, j, nx)] = c0
                A[k, idx(i, j - 1, nx)] = cx
                A[k, idx(i, j + 1, nx)] = cx
                A[k, idx(i - 1, j, nx)] = cy
                A[k, idx(i + 1, j, nx)] = cy

    sol = spsolve(A.tocsr(), b)
    return sol.reshape((ny, nx))

def compute_reduced_fields(Vsol):
    dV_dy, dV_dx = np.gradient(Vsol, dy, dx)
    Ex_red = -dV_dx
    Ey_red = -dV_dy
    E_red_mag = np.sqrt(Ex_red**2 + Ey_red**2)
    return Ex_red, Ey_red, E_red_mag

def compute_physical_pseudopotential(Vsol):
    Ex_red, Ey_red, E_red_mag = compute_reduced_fields(Vsol)
    Ex_phys = Ex_red / length_scale_m
    Ey_phys = Ey_red / length_scale_m
    E_phys_mag = np.sqrt(Ex_phys**2 + Ey_phys**2)
    Phi_phys_J = (e_charge**2 * E_phys_mag**2) / (4.0 * ion_mass * Omega**2)
    Phi_phys_eV = Phi_phys_J / e_charge
    return Ex_phys, Ey_phys, E_phys_mag, Phi_phys_J, Phi_phys_eV

def second_derivatives(F, i, j, dx_eff, dy_eff):
    d2F_dx2 = (F[i, j + 1] - 2.0 * F[i, j] + F[i, j - 1]) / dx_eff**2
    d2F_dy2 = (F[i + 1, j] - 2.0 * F[i, j] + F[i - 1, j]) / dy_eff**2
    d2F_dxdy = (
        F[i + 1, j + 1] - F[i + 1, j - 1] - F[i - 1, j + 1] + F[i - 1, j - 1]
    ) / (4.0 * dx_eff * dy_eff)
    return d2F_dx2, d2F_dy2, d2F_dxdy

def find_best_trap_in_window(Phi_phys_J):
    margin_j = 3
    y_min_frac = 0.10
    y_max_frac = 0.50
    i_min = max(2, int(y_min_frac * ny))
    i_max = min(ny - 3, int(y_max_frac * ny))

    best = None
    best_score = -np.inf

    for i in range(i_min, i_max):
        for j in range(margin_j, nx - margin_j):
            center = Phi_phys_J[i, j]
            neighbors = [
                Phi_phys_J[i - 1, j],
                Phi_phys_J[i + 1, j],
                Phi_phys_J[i, j - 1],
                Phi_phys_J[i, j + 1],
            ]
            if not all(center <= n for n in neighbors):
                continue

            d2x, d2y, d2xy = second_derivatives(Phi_phys_J, i, j, dx_phys, dy_phys)
            H = np.array([[d2x, d2xy], [d2xy, d2y]], dtype=float)
            eigvals, eigvecs = np.linalg.eigh(H)
            if np.min(eigvals) <= 0.0:
                continue

            trap_strength = float(np.sum(eigvals))
            trap_height_m = y[i] * length_scale_m
            objective_score = trap_strength / max(trap_height_m, 1e-12)

            if objective_score > best_score:
                best_score = objective_score
                best = {
                    'i': i,
                    'j': j,
                    'H_phys': H,
                    'eigvals_phys': eigvals,
                    'trap_strength': trap_strength,
                    'trap_height_m': trap_height_m,
                    'objective_score': objective_score,
                    'window': (i_min, i_max, margin_j),
                }
    return best

def evaluate_design(center_width, gap, v_rf, v_dc):
    left_mask, center_mask, right_mask, remaining_ground = make_electrode_masks(center_width, gap, side_width)
    V, fixed = build_boundary_problem(left_mask, center_mask, right_mask, remaining_ground, v_rf, v_dc)
    Vsol = solve_laplace(V, fixed)
    Ex_phys, Ey_phys, E_phys_mag, Phi_phys_J, Phi_phys_eV = compute_physical_pseudopotential(Vsol)
    best_trap = find_best_trap_in_window(Phi_phys_J)

    if best_trap is None:
        return {
            'valid': False,
            'center_width': center_width,
            'gap': gap,
            'v_rf': v_rf,
            'v_dc': v_dc,
            'left_mask': left_mask,
            'center_mask': center_mask,
            'right_mask': right_mask,
            'remaining_ground': remaining_ground,
            'Vsol': Vsol,
            'Phi_phys_J': Phi_phys_J,
            'Phi_phys_eV': Phi_phys_eV,
        }

    i = best_trap['i']
    j = best_trap['j']
    trap_x = x[j]
    trap_y = y[i]
    trap_x_m = trap_x * length_scale_m
    trap_y_m = trap_y * length_scale_m
    trap_value_eV = Phi_phys_eV[i, j]
    eigvals_phys = best_trap['eigvals_phys']
    omega_rad_s = np.sqrt(np.clip(eigvals_phys / ion_mass, 0.0, None))
    omega_hz = omega_rad_s / (2.0 * np.pi)

    return {
        'valid': True,
        'center_width': center_width,
        'gap': gap,
        'v_rf': v_rf,
        'v_dc': v_dc,
        'left_mask': left_mask,
        'center_mask': center_mask,
        'right_mask': right_mask,
        'remaining_ground': remaining_ground,
        'Vsol': Vsol,
        'Phi_phys_J': Phi_phys_J,
        'Phi_phys_eV': Phi_phys_eV,
        'trap_x': trap_x,
        'trap_y': trap_y,
        'trap_x_m': trap_x_m,
        'trap_y_m': trap_y_m,
        'trap_value_eV': trap_value_eV,
        'H_phys': best_trap['H_phys'],
        'eigvals_phys': eigvals_phys,
        'omega_hz': omega_hz,
        'trap_strength': best_trap['trap_strength'],
        'objective_score': best_trap['objective_score'],
        'window': best_trap['window'],
    }


## Combined design-space scan

We scan a compact combined design space and evaluate each candidate directly in physical units.

The design objective is:

$$
\text{score} = \frac{\text{trap strength}}{\text{trap height}}
$$

where trap strength is the sum of positive principal curvatures at the selected trap point.


In [ ]:
v_rf_values = np.array([0.8, 1.2, 1.6])
v_dc_values = np.array([-2.0, -1.5, -1.0])
center_width_values = np.array([0.6, 0.8, 1.0, 1.2])
gap_values = np.array([0.2, 0.35, 0.5, 0.65])

total_cases = len(v_rf_values) * len(v_dc_values) * len(center_width_values) * len(gap_values)
print(f'Scanning {total_cases} combined designs...')

results = []
for gap in gap_values:
    for center_width in center_width_values:
        for v_dc in v_dc_values:
            for v_rf in v_rf_values:
                results.append(evaluate_design(center_width, gap, v_rf, v_dc))

valid_results = [r for r in results if r['valid']]
print(f'Valid designs found: {len(valid_results)}')
if not valid_results:
    raise RuntimeError('No valid designs found in the combined scan.')


## Best global design


In [ ]:
best_global = max(valid_results, key=lambda r: r['objective_score'])

print('Best global design found.')
print(f"center_width = {best_global['center_width']:.3f}")
print(f"gap = {best_global['gap']:.3f}")
print(f"v_rf = {best_global['v_rf']:.3f}")
print(f"v_dc = {best_global['v_dc']:.3f}")
print(f"trap location = ({best_global['trap_x_m']*1e6:.1f} um, {best_global['trap_y_m']*1e6:.1f} um)")
print(f"trap pseudopotential at point = {best_global['trap_value_eV']:.3e} eV")
print(f"objective_score = {best_global['objective_score']:.3e}")
print('Approximate secular frequencies:')
for k, f in enumerate(best_global['omega_hz'], start=1):
    print(f'  mode {k}: {f/1e6:.3f} MHz')


## Aggregate map: best objective score by geometry


In [ ]:
best_score_map = np.zeros((len(gap_values), len(center_width_values)))
best_height_map = np.zeros((len(gap_values), len(center_width_values)))
best_freq_map = np.zeros((len(gap_values), len(center_width_values)))

for ig, gap in enumerate(gap_values):
    for iw, center_width in enumerate(center_width_values):
        group = [r for r in valid_results if np.isclose(r['gap'], gap) and np.isclose(r['center_width'], center_width)]
        if group:
            best_here = max(group, key=lambda r: r['objective_score'])
            best_score_map[ig, iw] = best_here['objective_score']
            best_height_map[ig, iw] = best_here['trap_y_m'] * 1e6
            best_freq_map[ig, iw] = np.max(best_here['omega_hz']) / 1e6

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(best_score_map, origin='lower', aspect='auto', extent=[center_width_values[0], center_width_values[-1], gap_values[0], gap_values[-1]])
ax.set_title('Best Objective Score vs Geometry Pair')
ax.set_xlabel('Center electrode width')
ax.set_ylabel('Gap')
fig.colorbar(im, ax=ax, label='Best score')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(best_height_map, origin='lower', aspect='auto', extent=[center_width_values[0], center_width_values[-1], gap_values[0], gap_values[-1]])
ax.set_title('Best Trap Height vs Geometry Pair')
ax.set_xlabel('Center electrode width')
ax.set_ylabel('Gap')
fig.colorbar(im, ax=ax, label='Trap height (um)')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(best_freq_map, origin='lower', aspect='auto', extent=[center_width_values[0], center_width_values[-1], gap_values[0], gap_values[-1]])
ax.set_title('Best Max Secular Frequency vs Geometry Pair')
ax.set_xlabel('Center electrode width')
ax.set_ylabel('Gap')
fig.colorbar(im, ax=ax, label='Max mode frequency (MHz)')
plt.tight_layout()
plt.show()


## Best global design: physical pseudopotential map


In [ ]:
X, Y = np.meshgrid(x, y)
X_um = X * length_scale_m * 1e6
Y_um = Y * length_scale_m * 1e6

i_min, i_max, margin_j = best_global['window']
x_left_um = x[margin_j] * length_scale_m * 1e6
x_right_um = x[nx - margin_j - 1] * length_scale_m * 1e6
y_bottom_um = y[i_min] * length_scale_m * 1e6
y_top_um = y[i_max - 1] * length_scale_m * 1e6

fig, ax = plt.subplots(figsize=(9, 5.5))
im = ax.pcolormesh(X_um, Y_um, best_global['Phi_phys_eV'], shading='auto')
cs = ax.contour(X_um, Y_um, best_global['Phi_phys_eV'], levels=15)
ax.clabel(cs, inline=True, fontsize=8)
ax.scatter([best_global['trap_x_m'] * 1e6], [best_global['trap_y_m'] * 1e6], s=90, label='Best global trap point')
ax.plot(x[best_global['left_mask']] * length_scale_m * 1e6, np.full(np.sum(best_global['left_mask']), 0.0), linewidth=7)
ax.plot(x[best_global['center_mask']] * length_scale_m * 1e6, np.full(np.sum(best_global['center_mask']), 0.0), linewidth=7)
ax.plot(x[best_global['right_mask']] * length_scale_m * 1e6, np.full(np.sum(best_global['right_mask']), 0.0), linewidth=7)
ax.plot([x_left_um, x_right_um, x_right_um, x_left_um, x_left_um], [y_bottom_um, y_bottom_um, y_top_um, y_top_um, y_bottom_um], linestyle='--', linewidth=1.5, label='Trap search window')
ax.set_title('Best Global Design: Physical RF-Style Pseudopotential')
ax.set_xlabel('x (um)')
ax.set_ylabel('y (um)')
ax.legend()
fig.colorbar(im, ax=ax, label='Pseudopotential (eV)')
plt.tight_layout()
plt.show()


## Best global design: local cuts


In [ ]:
min_i = int(np.argmin(np.abs(y * length_scale_m - best_global['trap_y_m'])))
min_j = int(np.argmin(np.abs(x * length_scale_m - best_global['trap_x_m'])))

j_window = slice(max(min_j - 8, 0), min(min_j + 9, nx))
i_window = slice(max(min_i - 8, 0), min(min_i + 9, ny))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x[j_window] * length_scale_m * 1e6, best_global['Phi_phys_eV'][min_i, j_window])
ax.axvline(best_global['trap_x_m'] * 1e6, linestyle='--', linewidth=1)
ax.set_title('Best Global Design: Local Cut Along x')
ax.set_xlabel('x (um)')
ax.set_ylabel('Pseudopotential (eV)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(y[i_window] * length_scale_m * 1e6, best_global['Phi_phys_eV'][i_window, min_j])
ax.axvline(best_global['trap_y_m'] * 1e6, linestyle='--', linewidth=1)
ax.set_title('Best Global Design: Local Cut Along y')
ax.set_xlabel('y (um)')
ax.set_ylabel('Pseudopotential (eV)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Hardware-facing design summary


In [ ]:
print('Hardware-Facing Design Summary')
print('------------------------------')
print(f"Geometry:")
print(f"  center_width = {best_global['center_width']:.3f}")
print(f"  gap = {best_global['gap']:.3f}")
print(f"  side_width = {side_width:.3f}")
print(f"Voltages:")
print(f"  v_rf = {best_global['v_rf']:.3f}")
print(f"  v_dc = {best_global['v_dc']:.3f}")
print(f"Trap location:")
print(f"  x = {best_global['trap_x_m']*1e6:.1f} um")
print(f"  y = {best_global['trap_y_m']*1e6:.1f} um")
print(f"Trap scale:")
print(f"  pseudopotential at trap point = {best_global['trap_value_eV']:.3e} eV")
print(f"  objective_score = {best_global['objective_score']:.3e}")
print(f"Approximate secular frequencies:")
for k, f in enumerate(best_global['omega_hz'], start=1):
    print(f'  mode {k}: {f/1e6:.3f} MHz')


## Notes and next steps

This notebook integrates the reduced optimization and physical-scaling workflows into a single end-to-end design pass.

Important caveats:

- the model is still 2D and simplified
- the RF pseudopotential is based on a reduced boundary-voltage description
- axial confinement and compensation fields are not yet included
- frequency estimates depend on the chosen physical length scale and RF frequency

Natural next steps:

- refine the scan around the best region with a denser grid
- compare multiple ion species and RF frequencies
- add explicit trap-depth diagnostics in eV
- refactor reusable solvers and evaluation functions into `src/`
- extend the reduced 2D workflow toward more realistic trapped-ion geometry models

That progression will move the workflow further toward a practical surface-electrode ion-trap design toolkit.
